# Desafio de modelagem · FGV

Treine um classificador com a base sintética rotulada e gere uma previsão para cada linha de dados/teste.csv. A meta proposta é 90% de acurácia no teste privado. O resultado público e a regra final de bonificação dependem da ativação do corretor pelo docente.

Este caderno é um ponto de partida, não uma solução. Registre seus experimentos, a divisão de validação, as métricas e a escolha do modelo. Use somente o treino para aprender transformações e selecionar variáveis.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from sklearn.ensemble import RandomForestClassifier

treino = pd.read_csv('dados/treino.csv')
teste = pd.read_csv('dados/teste.csv')
print('Treino:', treino.shape, '| Teste:', teste.shape)
print('Classe positiva:', round(treino['alto_desempenho'].mean(), 3))
treino.head(3)

## Validação interna

Divida apenas os 40 mil exemplos com resposta. O teste oculto não pode orientar a escolha de parâmetros. Experimente outras divisões e, se fizer muitas tentativas, mantenha uma validação final que não foi usada para escolher o modelo.

In [ ]:
X = treino.drop(columns=['id', 'alto_desempenho'])
y = treino['alto_desempenho']
X_dev, X_val, y_dev, y_val = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)
print('Desenvolvimento:', X_dev.shape, '| Validação:', X_val.shape)

In [ ]:
X_dev_num = pd.get_dummies(X_dev, dummy_na=True, dtype='int8')
X_val_num = pd.get_dummies(X_val, dummy_na=True, dtype='int8')
X_val_num = X_val_num.reindex(columns=X_dev_num.columns, fill_value=0)
medianas = X_dev_num.median(numeric_only=True)
X_dev_num = X_dev_num.fillna(medianas)
X_val_num = X_val_num.fillna(medianas)
print('Variáveis após preparo:', X_dev_num.shape[1])

In [ ]:
modelo = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=1)
modelo.fit(X_dev_num, y_dev)
prev_val = modelo.predict(X_val_num)
print('Acurácia:', round(accuracy_score(y_val, prev_val), 4))
print('F1:', round(f1_score(y_val, prev_val), 4))

## Seu experimento

Pesquise e compare classificadores do scikit-learn. Considere tratamento das ausências, escalas diferentes, transformação de variáveis, seleção e ajuste de hiperparâmetros. Se mudar o preparo, reproduza exatamente o mesmo procedimento ao treinar com toda a base e prever o teste.

In [ ]:
# Teste aqui outras abordagens e registre suas métricas.
# Exemplo: experimente um novo classificador ou transforme variáveis.
# Ao escolher o melhor caminho, adapte a próxima célula.


In [ ]:
X_tudo = pd.get_dummies(X, dummy_na=True, dtype='int8')
X_teste = pd.get_dummies(teste.drop(columns=['id']), dummy_na=True, dtype='int8')
X_teste = X_teste.reindex(columns=X_tudo.columns, fill_value=0)
medianas_finais = X_tudo.median(numeric_only=True)
X_tudo = X_tudo.fillna(medianas_finais)
X_teste = X_teste.fillna(medianas_finais)

# Substitua este modelo e este preparo pelo caminho escolhido na validação.
modelo_final = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=1)
modelo_final.fit(X_tudo, y)
entrega = pd.DataFrame({'id': teste['id'], 'previsao': modelo_final.predict(X_teste)})
entrega.to_csv('previsoes.csv', index=False)
print('Arquivo criado:', len(entrega), 'previsões')
entrega.head(3)